# Yaya-1B Training (Kaggle)
**Before running:** Settings → Accelerator → **GPU T4 x2** → Save

- Uses 8-bit AdamW (bitsandbytes) — cuts optimizer memory 8GB → 2GB
- bfloat16 precision + WSD schedule for training stability
- Checkpoints saved to `/kaggle/working/yaya-checkpoints-1b/`
- Download from the Output tab after the session ends

**Session flow:**
1. First session: tokenize Wikipedia + train as far as possible (~12h limit)
2. Subsequent sessions: Step 3 auto-resumes from latest checkpoint
3. After pretrain converges: run Step 4 (SFT) in a new session

In [ ]:
# Step 1: Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
if torch.cuda.device_count() > 1:
    print(f'  ({torch.cuda.device_count()} GPUs available)')

In [ ]:
# Step 2: Setup
import os
!git clone https://github.com/jaylink-coder/miss-yaya.git /kaggle/working/miss-yaya 2>/dev/null || \
    (cd /kaggle/working/miss-yaya && git pull)
# bitsandbytes: 8-bit AdamW — cuts optimizer memory 8GB → 2GB (essential for 1B on T4)
!pip install -q sentencepiece datasets pyyaml bitsandbytes
os.chdir('/kaggle/working/miss-yaya/yaya-ai')
print('Working dir:', os.getcwd())

In [ ]:
# Step 3: Pretrain Yaya-1B
# - First run: tokenizes all Wikipedia (~60 min), then trains
# - Re-runs: skips tokenization, auto-resumes from latest checkpoint
# - Checkpoints: /kaggle/working/yaya-checkpoints-1b/
!python scripts/kaggle_1b.py

In [ ]:
# Step 4: SFT — run after pretraining converges
# Downloads OpenHermes 100K + fine-tunes on combined SFT data
# Checkpoints: /kaggle/working/yaya-sft-1b-checkpoints/
!python scripts/kaggle_sft_1b.py

In [ ]:
# Step 5: Evaluate
import glob
sft_ckpts = sorted(glob.glob('/kaggle/working/yaya-sft-1b-checkpoints/checkpoint-*'))
best = sft_ckpts[-1] if sft_ckpts else None
if best:
    import os
    os.system(
        f'python scripts/eval_yaya.py '
        f'--model_config configs/model/yaya_1b.yaml '
        f'--checkpoint {best} --chat'
    )
else:
    print('No SFT checkpoint found.')